## 📘 Cats vs Dogs Image Classification using HOG + PCA + KNN


Course: CSC 304 – Machine Learning

University: SRM University AP


Team Members:

1. Datha Sai

2. Reshmi

3. Hemanth


## 1. Abstract

This project implements a machine learning model to classify images of cats and dogs using traditional ML techniques instead of deep learning. We use Histogram of Oriented Gradients (HOG) for feature extraction, Principal Component Analysis (PCA) for dimensionality reduction, and K-Nearest Neighbors (KNN) for classification. The goal is to build an explainable, syllabus-aligned image classifier and evaluate its performance using real-world data.

## 2. Introduction

Image classification is a fundamental problem in machine learning and computer vision. While deep learning approaches are widely used today, traditional ML methods can still perform well when combined with strong feature extraction.

This project builds a binary classifier that distinguishes between cats and dogs using a pipeline consisting of:

- HOG for extracting shape and edge-based features,

- PCA for compressing high-dimensional feature vectors, and

- KNN for classification.

The project demonstrates the practical application of classical ML algorithms taught in CSC 304.

In [34]:
# Modules

import numpy as np
import matplotlib.pyplot as plt
import cv2 
import os

from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report



In [35]:
# Paths

cat_path = "PetImages/Cat"
dog_path = "PetImages/Dog"


## 3. Dataset Description

We use a Cats vs Dogs dataset (Kaggle/Microsoft), which is outside the scikit-learn library.

Dataset Details

- Two classes: Cat (0) and Dog (1)

- Images of varying shapes and sizes

- Converted all images to:

  - 128×128 resolution

  - Grayscale

Train/Test Split

- 80% Training

- 20% Testing

Preprocessing Applied

1. Load images

2. Resize to 128×128

3. Convert to grayscale

4. Extract HOG descriptors

5. Store features in X, labels in y

In [ ]:
# Cleaning Data

def load_images_from_folder(folder, label):
    images = []
    labels = []

    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        try:
            img = cv2.imread(img_path)

            if img is None:
                continue

            img = cv2.resize(img, (128, 128)) 
            images.append(img)
            labels.append(label)

        except:
            pass

    return images, labels


cat_images, cat_labels = load_images_from_folder(cat_path, 0)  # 0 = Cat
dog_images, dog_labels = load_images_from_folder(dog_path, 1)  # 1 = Dog

print("Cats loaded:", len(cat_images))
print("Dogs loaded:", len(dog_images))


Cats loaded: 12498
Dogs loaded: 12499


## 4. Methodology
Workflow

1. Load dataset

2. Preprocess and normalize images

3. Extract HOG features

4. Reduce dimensions using PCA

5. Train KNN classifier

6. Evaluate model on test data

In [37]:
# Combines cats and dogs data

images = np.array(cat_images + dog_images)
labels = np.array(cat_labels + dog_labels)

print("Total images:", len(images))


Total images: 24997


## 5. HOG – Histogram of Oriented Gradients

HOG is a feature descriptor that captures the edge directions and gradients in images. Animals, especially cats and dogs, can be differentiated by their shapes, which makes HOG effective.

Why HOG?

- Captures structural information

- Robust to lighting changes

- Works well for object detection

- Provides compact numeric features

How HOG Works

1. Compute gradients in X and Y directions

2. Divide the image into small cells

3. Build orientation histograms

4. Normalize across blocks

5. Flatten into a feature vector

HOG converts images into meaningful numerical features that KNN can process.

In [ ]:
# HOG Feature Extraction


X = []
y = []

for img, label in zip(images, labels):

    img = cv2.resize(img, (128, 128))

    feature = hog(
        img,
        orientations=12,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        visualize=False,
        channel_axis=2
    )

    hist = cv2.calcHist(
        [img],            # must be BGR
        [0, 1, 2],        # channels
        None,             # mask
        [8, 8, 8],        # bins
        [0, 256, 0, 256, 0, 256]
    )
    hist = cv2.normalize(hist, hist).flatten()

    features = np.hstack([feature, hist])

    X.append(features)
    y.append(label)



In [39]:
# splitting data for train and test

X = np.array(X)
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 19997
Testing samples: 5000


## 6. PCA – Principal Component Analysis

HOG features are very high-dimensional (sometimes thousands of features per image). To make computation efficient, we apply PCA.

Why PCA?

- Reduces dimensionality

- Removes noise

- Speeds up classification

- Prevents overfitting

Approach

- PCA fitted on training data

- Retained 95% variance

- Transformed both train and test feature matrices

This reduces the feature size while retaining important information.

In [ ]:
# Applying PCA (principle application analysis) for dimensionality reduction

pca = PCA(n_components = 0.95)   # keep 95% variance
pca.fit(X_train)

X_train_pca = pca.transform(X_train)
X_test_pca  = pca.transform(X_test)

print("Original HOG feature dimensions:", X_train.shape[1])
print("Reduced PCA feature dimensions:", X_train_pca.shape[1])


Original HOG feature dimensions: 2864
Reduced PCA feature dimensions: 641


## 7. KNN – K-Nearest Neighbors

KNN is a simple, instance-based classifier that predicts a label based on the majority class among its nearest neighbors.

Why KNN?

- Easy to implement and understand

- No complex training required

- Works well with PCA-compressed feature vectors

- Perfect for traditional ML projects

Configuration Used

- k = 5

- Distance metric: Euclidean

- Weighted voting: uniform

In [41]:
# Applying KNN

best_acc = 0
best_k = 0

for k in range(1, 21):

    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_pca, y_train)   # TRAIN ON PCA

    y_pred = knn.predict(X_test_pca)  # TEST ON PCA

    acc = accuracy_score(y_test, y_pred)

    if acc > best_acc:
        best_acc = acc
        best_k = k

print("Best K =", best_k)
print("Best Accuracy =", best_acc)



Best K = 2
Best Accuracy = 0.6824


## 8. Experimental Results

We evaluate the model using:

- Accuracy Score

- Classification Report

- Confusion Matrix

Observations

- PCA significantly improves both speed and accuracy.

- KNN works effectively on PCA-compressed HOG data.

- The model distinguishes between cats and dogs with reasonably high accuracy.

In [42]:
# Evalutation

y_pred = knn.predict(X_test_pca)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.6344

Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.32      0.47      2515
           1       0.58      0.95      0.72      2485

    accuracy                           0.63      5000
   macro avg       0.73      0.64      0.60      5000
weighted avg       0.73      0.63      0.59      5000


Confusion Matrix:

[[ 807 1708]
 [ 120 2365]]


## 9. Discussion

- HOG successfully captures the shape differences between animals.

- PCA compresses data from thousands of features to manageable dimensions.

- KNN provides good performance without complex training.

- The combination of HOG + PCA + KNN is efficient and interpretable.

## 10. Conclusion

This project demonstrates a complete machine learning pipeline for image classification. By combining HOG, PCA, and KNN, we successfully built an explainable Cats vs Dogs classifier aligned with the Machine Learning curriculum.

Key takeaways:

- Feature extraction is critical in classical ML

- Dimensionality reduction improves performance

- Simple algorithms like KNN can be strong baselines

## 11. References

- Dalal, N., & Triggs, B. (2005). Histograms of Oriented Gradients for Human Detection.

- Scikit-image Documentation

- Scikit-learn Documentation

- Kaggle Cats vs Dogs Dataset

In [ ]:
 

img = cv2.imread("dog1.jpg")

img = cv2.resize(img, (128, 128))

feature = hog(
    img,
    orientations=12,
    pixels_per_cell=(16, 16),
    cells_per_block=(2, 2),
    block_norm="L2-Hys",
    visualize=False,
    channel_axis=2 
)

hist = cv2.calcHist(
    [img],            # must be BGR
    [0, 1, 2],        # channels B,G,R
    None,             # mask
    [8, 8, 8],        # bins
    [0, 256, 0, 256, 0, 256]
)
hist = cv2.normalize(hist, hist).flatten()
features = np.hstack([feature, hist])

pca_feature = pca.transform([features])
pred = knn.predict(pca_feature)[0]

print("Cat" if pred == 0 else "Dog")


Dog
